# 01 — WordCount, Local Mode

**What runs where:** the Jupyter kernel of your Notebook Pod (`jupyter-<user>` in namespace `user-<user>`) starts a JVM through Py4J. That JVM is the Spark **Driver**, and with `master("local[*]")` it also runs every task in its own threads. Kubernetes creates **no** Executor Pods.

**Observe:**
- Spark UI: `http://localhost:8080/spark-ui/<user>/` while the session is alive (the first cell prints the exact link). `http://jupyter-<user>:4040` (what `sc.uiWebUrl` and the `spark` object's "Spark UI" link show) is the in-cluster address and does not resolve in your browser.
- On the Mac: `kubectl -n user-<user> get pods` shows no `*-exec-*` Pods.

> This folder is refreshed from the runtime image each time the Notebook Server starts. Copy the notebook elsewhere under `work/` before editing it.

In [ ]:
import os, socket, threading
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("WordCount-Local")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
print("master       :", sc.master)
print("driver Pod   :", socket.gethostname())
print("kernel pid   :", os.getpid())
# sc.uiWebUrl (e.g. http://jupyter-alice:4040) is the in-cluster address; the
# browser reaches the same UI through the lab portal at /spark-ui/<user>/.
user = os.environ.get("SPARK_USER", "<user>")
ui_port = sc.uiWebUrl.rsplit(":", 1)[-1]
if ui_port == "4040":
    print(f"Spark UI     : http://localhost:8080/spark-ui/{user}/  (or http://<mac-ip>:8080/spark-ui/{user}/)")
else:
    print(f"Spark UI     : port {ui_port} is NOT forwarded; another SparkSession in this Pod already holds 4040."
          f" Stop it (or restart that kernel) to see this application at /spark-ui/{user}/.")

In [ ]:
lines = sc.parallelize(["spark kubernetes", "spark notebook", "pyspark spark"], 2)
counts = (
    lines.flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
)
print(sorted(counts.collect()))

## Where did the tasks run?

Each task reports its host name and Python worker PID. In Local Mode the host is always this Notebook Pod. The PIDs differ from the kernel PID because the Driver JVM forks separate Python worker processes for RDD lambdas.

In [ ]:
evidence = (
    sc.parallelize(range(4), 4)
    .map(lambda _: (socket.gethostname(), os.getpid()))
    .distinct()
    .collect()
)
for host, pid in sorted(evidence):
    print(f"host={host}  python-worker-pid={pid}")
print("JVM executors seen by the Driver:", sc._jsc.sc().getExecutorMemoryStatus().size(), "(1 = the Driver itself)")

In [ ]:
# Release the Driver (and port 4040). The /spark-ui/<user>/ link stops working after this.
spark.stop()